# CropYieldDataDownloader — EVA

Descarga registros agrícolas EVA publicados mediante Socrata y los guarda como Parquet crudo. La salida se particiona por fuente, departamento, año y **período EVA real** (`2022`, `2022A`, `2022B`); no convierte semestres o registros anuales en meses ficticios.

Este notebook solo descarga, valida el contrato mínimo y normaliza tipos básicos. La deduplicación, consolidación por cultivo y municipio, recálculo del rendimiento y selección del target pertenecen al curador EVA.

## 1. Configuración

La descarga usa códigos DANE de departamento para evitar diferencias de mayúsculas y acentos en los nombres publicados. Con `SOBRESCRIBIR_PARQUET=False`, cada período se reanuda desde el total de filas ya validado.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import re
import subprocess
import sys
import time
import unicodedata

import pandas as pd
import requests

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-15'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(
            ['git', 'fetch', '--depth', '1', 'origin', f'+refs/heads/{REPO_REF}:{remote_ref}'],
            cwd=REPO_DIR,
            check=True,
        )
        subprocess.run(
            ['git', 'checkout', '-B', REPO_REF, remote_ref],
            cwd=REPO_DIR,
            check=True,
        )
    PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
else:
    candidatos = [
        REPO_DIR / 'notebooks' / 'ClimatePipeline',
        REPO_DIR / 'ClimatePipeline',
        REPO_DIR,
        REPO_DIR.parent / 'ClimatePipeline',
    ]
    PIPELINE_DIR = next(
        (ruta for ruta in candidatos if (ruta / 'DatasetConfig.py').exists()),
        None,
    )
if PIPELINE_DIR is None or not PIPELINE_DIR.exists():
    raise FileNotFoundError('No se encontró notebooks/ClimatePipeline/DatasetConfig.py.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from DatasetConfig import cargar_configuracion_datasets

DATASET_CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
PROCESSED_ROOT = DATASET_CONFIG.processed_root

DATASET_ID = 'uejq-wxrr'
VARIABLE_NOMBRE = 'rendimientos_cultivo'
DESCARGA_DEPARTAMENTOS = ['BOYACÁ', 'CUNDINAMARCA']
DESCARGA_ANIOS = [2022, 2023, 2024, 2025]

CODIGOS_DANE_DEPARTAMENTO = {
    'BOYACÁ': '15',
    'CUNDINAMARCA': '25',
}

DESCARGA_LIMIT = 1000
DESCARGA_MAX_LOTES = None
MOSTRAR_CADA_N_LOTES = 10
SOBRESCRIBIR_PARQUET = False
REQUEST_TIMEOUT = 180
REQUEST_REINTENTOS = 3
APP_TOKEN = None  # Opcional. No subir tokens reales al repositorio.
EJECUTAR_DESCARGA = True

EVA_RAW_ROOT = DATASET_CONFIG.eva_raw_root

print({
    'dataset_id': DATASET_ID,
    'variable': VARIABLE_NOMBRE,
    'departamentos': DESCARGA_DEPARTAMENTOS,
    'anios': DESCARGA_ANIOS,
    'limit': DESCARGA_LIMIT,
    'max_lotes': DESCARGA_MAX_LOTES,
    'sobrescribir': SOBRESCRIBIR_PARQUET,
    'ejecutar': EJECUTAR_DESCARGA,
    'eva_raw_root': str(EVA_RAW_ROOT),
})

## 2. Contrato EVA y acceso a Socrata

In [ ]:
DATASET_ID_PATTERN = re.compile(r'^[a-z0-9]{4}-[a-z0-9]{4}$', re.IGNORECASE)
PART_PATTERN = re.compile(r'^part-(\d{5})\.parquet$')
PERIODO_PATTERN = re.compile(r'^\d{4}(?:A|B)?$')

CAMPOS_EVA_OBLIGATORIOS = {
    'c_digo_dane_departamento',
    'departamento',
    'c_digo_dane_municipio',
    'municipio',
    'cultivo',
    'a_o',
    'periodo',
    'rea_sembrada',
    'rea_cosechada',
    'producci_n',
    'rendimiento',
}
COLUMNAS_NUMERICAS_EVA = [
    'rea_sembrada',
    'rea_cosechada',
    'producci_n',
    'rendimiento',
]


def validar_configuracion():
    if not DATASET_ID_PATTERN.fullmatch(str(DATASET_ID)):
        raise ValueError('DATASET_ID debe tener el formato xxxx-xxxx.')
    if not DESCARGA_DEPARTAMENTOS:
        raise ValueError('Debe configurar al menos un departamento.')
    departamentos = sorted(set(str(d).strip().upper() for d in DESCARGA_DEPARTAMENTOS))
    desconocidos = set(departamentos) - set(CODIGOS_DANE_DEPARTAMENTO)
    if desconocidos:
        raise ValueError(f'No hay código DANE configurado para: {sorted(desconocidos)}')
    anios = sorted(set(int(a) for a in DESCARGA_ANIOS))
    if not anios or any(a < 1900 or a > 2100 for a in anios):
        raise ValueError(f'Años inválidos: {anios}')
    if int(DESCARGA_LIMIT) <= 0:
        raise ValueError('DESCARGA_LIMIT debe ser positivo.')
    if DESCARGA_MAX_LOTES is not None and int(DESCARGA_MAX_LOTES) <= 0:
        raise ValueError('DESCARGA_MAX_LOTES debe ser None o positivo.')
    return departamentos, anios


def headers_socrata():
    headers = {
        'Accept': 'application/json',
        'User-Agent': 'RAIZ-CropYieldDataDownloader/1.0',
    }
    if APP_TOKEN:
        headers['X-App-Token'] = APP_TOKEN
    return headers


def solicitar_json(url, params=None):
    ultimo_error = None
    for intento in range(1, REQUEST_REINTENTOS + 1):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers_socrata(),
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            return response.json()
        except requests.RequestException as exc:
            ultimo_error = exc
            if intento == REQUEST_REINTENTOS:
                break
            espera = min(2 ** (intento - 1), 30)
            print(f'Intento {intento}/{REQUEST_REINTENTOS} falló: {exc}; espera={espera}s')
            time.sleep(espera)
    raise ultimo_error


def consultar_metadata(dataset_id):
    return solicitar_json(f'https://www.datos.gov.co/api/views/{dataset_id}')


def validar_esquema_eva(metadata):
    campos = {columna.get('fieldName') for columna in metadata.get('columns', [])}
    faltantes = CAMPOS_EVA_OBLIGATORIOS - campos
    if faltantes:
        raise ValueError(f'El dataset no cumple el contrato EVA. Faltan: {sorted(faltantes)}')
    return campos


def consultar_query(dataset_id, query):
    url = f'https://www.datos.gov.co/resource/{dataset_id}.json'
    return solicitar_json(url, params={'$query': query})


def where_anio(codigo_departamento, anio):
    codigo = str(codigo_departamento).replace("'", "''")
    return f"c_digo_dane_departamento = '{codigo}' AND a_o = {int(anio)}"


def periodos_disponibles(dataset_id, codigo_departamento, anio):
    where = where_anio(codigo_departamento, anio)
    query = (
        f'SELECT periodo, count(*) AS total WHERE {where} '
        'GROUP BY periodo ORDER BY periodo'
    )
    filas = consultar_query(dataset_id, query)
    periodos = []
    for fila in filas:
        periodo = str(fila['periodo']).strip().upper()
        if not PERIODO_PATTERN.fullmatch(periodo):
            raise ValueError(f'Período EVA inesperado: {periodo!r}')
        if not periodo.startswith(str(int(anio))):
            raise ValueError(f'Período {periodo!r} no corresponde al año {anio}.')
        periodos.append({'periodo': periodo, 'filas_esperadas': int(fila['total'])})
    return periodos


def consultar_lote(dataset_id, codigo_departamento, anio, periodo, limit, offset):
    periodo_sql = str(periodo).replace("'", "''")
    where = f"{where_anio(codigo_departamento, anio)} AND periodo = '{periodo_sql}'"
    order = 'c_digo_dane_municipio, c_digo_del_cultivo, cultivo, :id'
    query = (
        f'SELECT * WHERE {where} ORDER BY {order} '
        f'LIMIT {int(limit)} OFFSET {int(offset)}'
    )
    return pd.DataFrame(consultar_query(dataset_id, query))

## 3. Particiones, normalización y reanudación

La cantidad publicada por Socrata se compara con la suma de filas de los Parquet locales. La descarga solo se marca completa cuando ambas cantidades coinciden.

In [ ]:
def valor_periodo_particion(valor):
    texto = str(valor).strip().upper()
    if not re.fullmatch(r'[A-Z0-9_-]+', texto):
        raise ValueError(f'Período de partición inseguro: {valor!r}')
    return texto


def departamento_para_ruta(departamento):
    texto = unicodedata.normalize('NFC', str(departamento).strip().upper())
    texto = re.sub(r'\s+', '_', texto)
    if not texto or texto in {'.', '..'} or '/' in texto or '\\' in texto:
        raise ValueError(f'Departamento de partición inseguro: {departamento!r}')
    return texto


def ruta_particion(dataset_id, departamento, anio, periodo):
    return (
        EVA_RAW_ROOT
        / f'fuente={str(dataset_id).lower()}'
        / f'departamento={departamento_para_ruta(departamento)}'
        / f'anio={int(anio)}'
        / f'periodo={valor_periodo_particion(periodo)}'
    )


def partes_existentes(output_dir):
    partes = []
    if not output_dir.exists():
        return partes
    for archivo in output_dir.glob('part-*.parquet'):
        match = PART_PATTERN.fullmatch(archivo.name)
        if match:
            partes.append((int(match.group(1)), archivo))
    partes.sort(key=lambda item: item[0])
    indices = [indice for indice, _ in partes]
    if indices != list(range(len(indices))):
        raise RuntimeError(f'Partes no consecutivas en {output_dir}: {indices}')
    return partes


def filas_en_partes(partes, limit):
    import pyarrow.parquet as pq

    filas = [pq.ParquetFile(archivo).metadata.num_rows for _, archivo in partes]
    for posicion, cantidad in enumerate(filas[:-1]):
        if cantidad != int(limit):
            raise RuntimeError(
                f'La parte {posicion} tiene {cantidad} filas; se esperaban {limit}.'
            )
    if filas and (filas[-1] <= 0 or filas[-1] > int(limit)):
        raise RuntimeError(f'Última parte inválida: {filas[-1]} filas.')
    return filas


def normalizar_lote(df, dataset_id):
    salida = df.copy()
    for columna in ['c_digo_dane_departamento', 'c_digo_dane_municipio']:
        if columna in salida.columns:
            ancho = 2 if columna == 'c_digo_dane_departamento' else 5
            salida[columna] = salida[columna].astype('string').str.zfill(ancho)
    salida['a_o'] = pd.to_numeric(salida['a_o'], errors='raise').astype('Int64')
    salida['periodo'] = salida['periodo'].astype('string').str.strip().str.upper()
    for columna in COLUMNAS_NUMERICAS_EVA:
        salida[columna] = pd.to_numeric(salida[columna], errors='coerce').astype('Float64')
    salida['dataset_id'] = str(dataset_id).lower()
    return salida


def validar_lote(df, codigo_departamento, anio, periodo):
    if df.empty:
        return
    codigos = set(df['c_digo_dane_departamento'].astype(str))
    anios = set(pd.to_numeric(df['a_o'], errors='raise').astype(int))
    periodos = set(df['periodo'].astype(str).str.upper())
    if codigos != {str(codigo_departamento)}:
        raise RuntimeError(f'El lote contiene otros departamentos: {sorted(codigos)}')
    if anios != {int(anio)}:
        raise RuntimeError(f'El lote contiene otros años: {sorted(anios)}')
    if periodos != {str(periodo).upper()}:
        raise RuntimeError(f'El lote contiene otros períodos: {sorted(periodos)}')


def descargar_periodo(
    dataset_id,
    departamento,
    codigo_departamento,
    anio,
    periodo,
    filas_esperadas,
    limit=1000,
    max_lotes=None,
    mostrar_cada_n_lotes=10,
    sobrescribir=False,
):
    inicio = time.perf_counter()
    output_dir = ruta_particion(dataset_id, departamento, anio, periodo)
    output_dir.mkdir(parents=True, exist_ok=True)
    partes = partes_existentes(output_dir)

    if sobrescribir and partes:
        for _, archivo in partes:
            archivo.unlink()
        partes = []

    filas_partes = filas_en_partes(partes, limit)
    filas_inicio = sum(filas_partes)
    if filas_inicio > int(filas_esperadas):
        raise RuntimeError(
            f'Hay {filas_inicio} filas locales, más que las {filas_esperadas} publicadas.'
        )
    if filas_inicio == int(filas_esperadas):
        return {
            'dataset_id': dataset_id,
            'departamento': departamento,
            'codigo_departamento': codigo_departamento,
            'anio': int(anio),
            'periodo': periodo,
            'estado': 'ya_completa',
            'filas_esperadas': int(filas_esperadas),
            'filas_inicio': filas_inicio,
            'filas_descargadas': 0,
            'filas_finales': filas_inicio,
            'partes_escritas': 0,
            'duracion_segundos': round(time.perf_counter() - inicio, 2),
            'carpeta': str(output_dir),
        }

    offset = filas_inicio
    parte_idx = len(partes)
    filas_descargadas = 0
    partes_escritas = 0
    estado = 'completa'

    while offset < int(filas_esperadas):
        if max_lotes is not None and partes_escritas >= int(max_lotes):
            estado = 'pausada_por_max_lotes'
            break
        df_lote = consultar_lote(
            dataset_id,
            codigo_departamento,
            anio,
            periodo,
            limit,
            offset,
        )
        if df_lote.empty:
            raise RuntimeError(
                f'Socrata devolvió 0 filas en offset={offset}, antes de {filas_esperadas}.'
            )
        validar_lote(df_lote, codigo_departamento, anio, periodo)
        df_lote = normalizar_lote(df_lote, dataset_id)
        output_path = output_dir / f'part-{parte_idx:05d}.parquet'
        temporal = output_dir / f'.{output_path.name}.tmp'
        df_lote.to_parquet(temporal, index=False, engine='pyarrow')
        temporal.replace(output_path)

        cantidad = len(df_lote)
        filas_descargadas += cantidad
        partes_escritas += 1
        offset += cantidad
        parte_idx += 1
        if partes_escritas == 1 or partes_escritas % int(mostrar_cada_n_lotes) == 0:
            print(
                f'{departamento} {periodo}: {offset:,}/{int(filas_esperadas):,} filas'
            )

    filas_finales = filas_inicio + filas_descargadas
    if estado == 'completa' and filas_finales != int(filas_esperadas):
        raise RuntimeError(
            f'Conteo final {filas_finales} != esperado {filas_esperadas}.'
        )
    return {
        'dataset_id': dataset_id,
        'departamento': departamento,
        'codigo_departamento': codigo_departamento,
        'anio': int(anio),
        'periodo': periodo,
        'estado': estado,
        'filas_esperadas': int(filas_esperadas),
        'filas_inicio': filas_inicio,
        'filas_descargadas': filas_descargadas,
        'filas_finales': filas_finales,
        'partes_escritas': partes_escritas,
        'duracion_segundos': round(time.perf_counter() - inicio, 2),
        'carpeta': str(output_dir),
    }

## 4. Plan y ejecución

In [ ]:
departamentos, anios = validar_configuracion()
metadata = consultar_metadata(DATASET_ID)
campos_dataset = validar_esquema_eva(metadata)

plan_descarga = []
sin_periodos = []
for departamento in departamentos:
    codigo = CODIGOS_DANE_DEPARTAMENTO[departamento]
    for anio in anios:
        periodos = periodos_disponibles(DATASET_ID, codigo, anio)
        if not periodos:
            sin_periodos.append({'departamento': departamento, 'anio': anio})
        for item in periodos:
            plan_descarga.append({
                'departamento': departamento,
                'codigo_departamento': codigo,
                'anio': anio,
                **item,
            })

vista_plan = pd.DataFrame(plan_descarga)
display(Markdown(
    f'### {metadata.get("name", DATASET_ID)} — '
    f'{len(plan_descarga)} particiones EVA'
))
display(vista_plan)
if sin_periodos:
    print('Combinaciones sin períodos publicados:', sin_periodos)

In [ ]:
resumen_particiones = pd.DataFrame()

if not EJECUTAR_DESCARGA:
    print('Descarga desactivada. Revise el plan y active EJECUTAR_DESCARGA.')
else:
    resumenes = []
    for indice, item in enumerate(plan_descarga, start=1):
        print(
            f'[{indice}/{len(plan_descarga)}] '
            f'{item["departamento"]} | {item["periodo"]} | '
            f'{item["filas_esperadas"]:,} filas esperadas'
        )
        try:
            resumen = descargar_periodo(
                dataset_id=DATASET_ID,
                departamento=item['departamento'],
                codigo_departamento=item['codigo_departamento'],
                anio=item['anio'],
                periodo=item['periodo'],
                filas_esperadas=item['filas_esperadas'],
                limit=DESCARGA_LIMIT,
                max_lotes=DESCARGA_MAX_LOTES,
                mostrar_cada_n_lotes=MOSTRAR_CADA_N_LOTES,
                sobrescribir=SOBRESCRIBIR_PARQUET,
            )
        except Exception as exc:
            resumen = {
                'dataset_id': DATASET_ID,
                **item,
                'estado': 'error',
                'error': f'{type(exc).__name__}: {exc}',
            }
            print(resumen['error'])
        resumenes.append(resumen)
    resumen_particiones = pd.DataFrame(resumenes)
    display(Markdown('## Resumen final'))
    display(resumen_particiones)

## 5. Verificación de salida

In [ ]:
if resumen_particiones.empty:
    print('No se ejecutaron particiones.')
else:
    errores = resumen_particiones[resumen_particiones['estado'] == 'error']
    incompletas = resumen_particiones[
        ~resumen_particiones['estado'].isin(['completa', 'ya_completa'])
    ]
    if not errores.empty:
        raise RuntimeError(f'Fallaron {len(errores)} particiones EVA.')
    if not incompletas.empty:
        print('Particiones pausadas o incompletas:')
        display(incompletas)
    else:
        total_esperado = int(resumen_particiones['filas_esperadas'].sum())
        total_final = int(resumen_particiones['filas_finales'].sum())
        if total_final != total_esperado:
            raise RuntimeError(f'Total local {total_final} != esperado {total_esperado}.')
        print(f'Verificación correcta: {total_final:,} filas en {len(resumen_particiones)} particiones.')
        print(f'Ruta: {EVA_RAW_ROOT}')
        print(f'Fin UTC: {datetime.now(timezone.utc).isoformat(timespec="seconds")}')

## 6. Siguiente etapa: curación EVA

La salida de este notebook aún es cruda. El curador EVA debe:

- normalizar códigos DANE como texto y verificar nombres geográficos;
- revisar duplicados por municipio, cultivo, desagregación, año y período;
- validar ciclo y estado físico del cultivo;
- consolidar producción y área solo entre categorías compatibles;
- recalcular rendimiento como producción total / área cosechada total;
- documentar el cambio metodológico de la fuente desde 2022.